In [1]:
import pandas as pd
df = pd.read_pickle("complete_dataset_labeled.pkl")
print(df.shape)
df.head(100)
df_cpu = df[df['Interference_Category'] == 'CPU'].copy()

print(df_cpu.shape)
df_cpu.head(100)

(1909, 19)
(356, 19)


,UniqueID,Test_ID,Replicas,Interference_Name,Interference_ID,Given_RPS,FolderID,Throughput,Avg_Latency,P50_Latency,P75_Latency,P90_Latency,P95_Latency,P99_Latency,Max_Latency,Errors,Interference_Category,norm_perf,Scenario_Label
26,1replicas_scenario11_1000rps_Fight_Club_V01,1replicas_scenario11_1000rps,1,1_iBench_CPU_pod,11,1000,Fight_Club_V01,999.999817,0.095,0.084,0.0,0.102,0.111,0.188,3.957,0.0,CPU,0.766548,CPU1
27,1replicas_scenario11_1000rps_Memento_V01,1replicas_scenario11_1000rps,1,1_iBench_CPU_pod,11,1000,Memento_V01,1000.004199,0.098,0.085,0.0,0.102,0.112,0.187,4.857,0.0,CPU,0.770648,CPU1
28,1replicas_scenario11_1000rps_TheGame_V01,1replicas_scenario11_1000rps,1,1_iBench_CPU_pod,11,1000,TheGame_V01,1000.002916,0.089,0.086,0.0,0.103,0.111,0.145,3.252,0.0,CPU,0.993870,CPU1
29,1replicas_scenario11_100rps_Memento_V01,1replicas_scenario11_100rps,1,1_iBench_CPU_pod,11,100,Memento_V01,100.005151,0.337,0.314,0.0,0.501,0.562,1.756,3.351,0.0,CPU,0.342635,CPU1
30,1replicas_scenario11_100rps_TheGame_V01,1replicas_scenario11_100rps,1,1_iBench_CPU_pod,11,100,TheGame_V01,100.005318,0.290,0.301,0.0,0.415,0.522,0.594,2.238,0.0,CPU,1.012907,CPU1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,1replicas_scenario14_3500rps_Fight_Club_V01,1replicas_scenario14_3500rps,1,4_iBench_CPU_pods,14,3500,Fight_Club_V01,3500.004211,0.316,0.195,0.0,0.480,1.007,2.713,8.410,0.0,CPU,0.075275,CPU4
122,1replicas_scenario14_3500rps_Memento_V01,1replicas_scenario14_3500rps,1,4_iBench_CPU_pods,14,3500,Memento_V01,3500.004778,0.187,0.109,0.0,0.254,0.370,2.237,7.066,0.0,CPU,0.091293,CPU4
123,1replicas_scenario14_3500rps_TheGame_V01,1replicas_scenario14_3500rps,1,4_iBench_CPU_pods,14,3500,TheGame_V01,3500.017767,0.184,0.169,0.0,0.292,0.338,0.491,4.242,0.0,CPU,0.415931,CPU4
124,1replicas_scenario14_4000rps_Fight_Club_V01,1replicas_scenario14_4000rps,1,4_iBench_CPU_pods,14,4000,Fight_Club_V01,4000.001290,0.207,0.119,0.0,0.309,0.484,2.423,7.893,0.0,CPU,0.100289,CPU4


In [2]:
df_baseline = df[df['Interference_Category'] == 'Baseline'].copy()

print(df_baseline.shape)
df_baseline.head()

(267, 19)


,UniqueID,Test_ID,Replicas,Interference_Name,Interference_ID,Given_RPS,FolderID,Throughput,Avg_Latency,P50_Latency,P75_Latency,P90_Latency,P95_Latency,P99_Latency,Max_Latency,Errors,Interference_Category,norm_perf,Scenario_Label
0,1replicas_scenario0_1000rps_Fight_Club_V01,1replicas_scenario0_1000rps,1,Baseline0,0,1000,Fight_Club_V01,1000.004331,0.087,0.084,0.0,0.100,0.108,0.141,2.682,0.0,Baseline,1.0,B0
1,1replicas_scenario0_1000rps_Memento_V01,1replicas_scenario0_1000rps,1,Baseline0,0,1000,Memento_V01,1000.002275,0.087,0.084,0.0,0.100,0.108,0.150,3.678,0.0,Baseline,1.0,B0
2,1replicas_scenario0_1000rps_TheGame_V01,1replicas_scenario0_1000rps,1,Baseline0,0,1000,TheGame_V01,1000.003594,0.088,0.085,0.0,0.102,0.111,0.146,3.211,0.0,Baseline,1.0,B0
3,1replicas_scenario0_100rps_Memento_V01,1replicas_scenario0_100rps,1,Baseline0,0,100,Memento_V01,100.004986,0.301,0.312,0.0,0.480,0.541,0.598,1.005,0.0,Baseline,1.0,B0
4,1replicas_scenario0_100rps_TheGame_V01,1replicas_scenario0_100rps,1,Baseline0,0,100,TheGame_V01,100.005097,0.295,0.310,0.0,0.442,0.539,0.606,2.918,0.0,Baseline,1.0,B0


In [4]:
# 1. Get unique values for both RPS and Replicas
unique_rps = df_cpu['Given_RPS'].unique()
unique_replicas = df_cpu['Replicas'].unique()

# 2. Create "Isolation" rows for EVERY combination of RPS and Replicas
isolation_rows = []
for rps in unique_rps:
    for rep in unique_replicas:
        isolation_rows.append({
            'Scenario_Label': 'Isolation',
            'Given_RPS': rps,
            'Replicas': rep,
            'norm_perf': 1.0,
            'Interference_Category': 'CPU'
        })

df_isolation = pd.DataFrame(isolation_rows)

# 3. Combine
df_cpu_isolation = pd.concat([df_cpu, df_isolation], ignore_index=True)

In [32]:
import plotly.graph_objects as go
import pandas as pd

# 1. Filter and sort
target_rps = [500, 1000, 1500, 2000]
plot_df = df_cpu_isolation[df_cpu_isolation['Given_RPS'].isin(target_rps)].copy()

# 2. Pivot the data into a grid format
# This aggregates (takes the mean) across all replicas automatically
pivot_df = plot_df.pivot_table(
    index='Given_RPS', 
    columns='Scenario_Label', 
    values='norm_perf', 
    aggfunc='mean'
)
column_order = ['Isolation', 'CPU1', 'CPU2', 'CPU3', 'CPU4']
# Filter to only columns that actually exist in pivot_df
existing_columns = [c for c in column_order if c in pivot_df.columns]
pivot_df = pivot_df[existing_columns]

# 3. Create the Figure with a single Aggregate Surface
fig = go.Figure()

fig.add_trace(go.Surface(
    z=pivot_df.values,
    x=pivot_df.columns,
    y=pivot_df.index,
    name='Aggregate NPS',
    colorscale='Viridis',  # Using a single clear color scale
    colorbar=dict(
        title='Aggregate NPS',
        tickfont=dict(size=18)
    ),
    cmin=0.1,
    cmax=1.0
))

# 4. Update layout with custom 3D annotations acting as spaced-out titles
fig.update_layout(
    title=dict(text='Aggregate NPS Performance Landscape', font=dict(size=30), x=0.5),
    width=1400,
    height=1000,
    scene=dict(
        # Hide the native titles so they don't overlap with our new ones
        xaxis=dict(title=dict(text=''), tickfont=dict(size=18)),
        yaxis=dict(title=dict(text=''), tickfont=dict(size=18)),
        zaxis=dict(title=dict(text=''), range=[0.1, 1], tickfont=dict(size=18)),
        
        # Manually position our own text labels using data grid coordinates
        annotations=[
            # 1. X-Axis Title ("Interference Scenarios")
            dict(
                showarrow=False,
                text="Interference Scenarios",
                x=2.0,          # Centers it at the middle category index (CPU2)
                y=300,        # PUSHES TEXT OUTWARD (Lower than your lowest 500 RPS)
                z=0.4,       # Keeps it at the base floor of the plot
                font=dict(size=24, color="black")
            ),
            # 2. Y-Axis Title ("Traffic Load (rps)")
            dict(
                showarrow=False,
                text="Traffic Load (rps)",
                x=-2.5,       # PUSHES THE TITLE LEFT, away from the Z values
                y=500,       # Keeps it aligned along the side plane
                z=0.55,        # Centers it vertically alongside the 0.1 to 1.0 grid
                font=dict(size=24, color="black")
            ),
            # 3. Z-Axis Title ("Aggregate NPS")
            dict(
                showarrow=False,
                text="Aggregate NPS",
                x=-2.5,       # PUSHES THE TITLE LEFT, away from the Z values
                y=500,       # Keeps it aligned along the side plane
                z=0.55,        # Centers it vertically alongside the 0.1 to 1.0 grid
                font=dict(size=24, color="black")
            )
        ],
        aspectmode='manual',
        aspectratio=dict(x=1.8, y=1, z=0.7)
    ),
    margin=dict(l=150, r=100, b=100, t=80) # Left margin padded out so Z title fits nicely
)

fig.show()

In [67]:
import plotly.graph_objects as go
import pandas as pd

# 1. Filter and sort
target_rps = [500, 1000, 1500, 2000]
plot_df = df_cpu_isolation[df_cpu_isolation['Given_RPS'].isin(target_rps)].copy()

# 2. Pivot the data into a grid format
# This aggregates (takes the mean) across all replicas automatically
pivot_df = plot_df.pivot_table(
    index='Given_RPS', 
    columns='Scenario_Label', 
    values='norm_perf', 
    aggfunc='mean'
)
column_order = ['Isolation', 'CPU1', 'CPU2', 'CPU3', 'CPU4']
# Filter to only columns that actually exist in pivot_df
existing_columns = [c for c in column_order if c in pivot_df.columns]
pivot_df = pivot_df[existing_columns]

# 3. Create the Figure with a single Aggregate Surface
fig = go.Figure()

fig.add_trace(go.Surface(
    z=pivot_df.values,
    x=pivot_df.columns,
    y=pivot_df.index,
    name='Aggregate NPS',
    colorscale='Viridis',  # Using a single clear color scale
    colorbar=dict(
        title='Aggregate NPS',
        tickfont=dict(size=18)
    ),
    cmin=0.1,
    cmax=1.0
))

# 4. Update layout with updated Aggregate NPS labels

# 4. Update layout using native 3D scene properties
fig.update_layout(
    title=dict(text='Aggregate NPS Performance Landscape', font=dict(size=30), x=0.5),
    width=1600,   # Canvas width adjusted for broader aspect ratio
    height=1200,
    scene=dict(
        xaxis=dict(
            title=dict(text='Interference Scenarios', font=dict(size=24)),
            # 1. Native mapping to inject spaces and cleanly push the title out
            tickvals=['Isolation', 'CPU1', 'CPU2', 'CPU3', 'CPU4'],
            ticktext=['Isolation   ', 'CPU1   ', 'CPU2   ', 'CPU3   ', 'CPU4   '],
            tickfont=dict(size=17)
        ),
        yaxis=dict(
            title=dict(text='Traffic Load (rps)', font=dict(size=24)),
            # 2. Add trailing breaks directly inside the title string to create space
            tickvals=[500, 1000, 1500, 2000],
            ticktext=['500   ', '1000   ', '1500   ', '2000   '],
            tickfont=dict(size=17)
        ),
        zaxis=dict(
            title=dict(text='Aggregate NPS', font=dict(size=24)),
            range=[0.1, 1],
            # 3. Add invisible space padding to separate the vertical label
            tickvals=[0.2, 0.4, 0.6, 0.8, 1.0],
            ticktext=['   0.2', '   0.4', '   0.6', '   0.8', '   1.0'],
            tickfont=dict(size=17)
        ),
        aspectmode='manual',
        # STRETCHES THE SCENARIOS AXIS: Increased x from 1 to 1.8
        aspectratio=dict(x=1.2, y=1, z=0.7) 
    ),
    margin=dict(l=150, r=100, b=100, t=80)
)



fig.show()